# 06d — Audit + experiments to beat majority voting (G=28)

Builds on **06c** (loads its cached G=28 artifacts). Two parts:
- **Deliverable 2 — Audit**: mechanistic diagnostics of *why* the alignment-based barycenter does not
  beat the symbol-frequency baselines on this data.
- **Deliverable 3 — Experiments**: honest attempts to beat `majority_voting` (transition/ordering features,
  Edit-Shape DTW, re-selected hyperparameters for the p_match feature, soft-mode DBA, length & mean-vs-mode
  ablations). Success = BH-significant paired-Wilcoxon win over majority voting. **No p-hacking.**

In [ ]:
%load_ext autoreload
%autoreload 2
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
import smartflat
from joblib import load
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.constants import incomplete_clinical_administrations
from smartflat.features.symbolization.utils import fix_clinical_diagnosis
from smartflat.features.symbolic_barycenter import vocab
from smartflat.features.symbolic_barycenter import baselines as B
from smartflat.features.symbolic_barycenter.baselines import make_patient_control_labels
from smartflat.engine.distances._rtwe import rtwe_pairwise_distance, rtwe_alignment_path
plt.rcParams['figure.dpi'] = 110

ANN, RND, CFG = 'samperochon', 8, 'SymbolicSourceInferenceGoldConfig'
L, NU, LMBDA, OFFSET = 64, 1e-4, 0.1, 0.3
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'g28')
AUD = os.path.join(OUT, 'audit'); EXP = os.path.join(OUT, 'experiments')
os.makedirs(AUD, exist_ok=True); os.makedirs(EXP, exist_ok=True)


In [ ]:
# --- load cached G=28 artifacts from 06c and rebuild the analysis cohort ---
dff = load(os.path.join(OUT, 'g28_full_with_cat.pkl'))
D_raw = np.load(os.path.join(OUT, 'D_G_cat_temporal_raw.npy'))
code = json.load(open(os.path.join(OUT, 'category_codes.json')))
code_to_label = [None]*len(code)
for k, v in code.items():
    code_to_label[v] = k
D_G_cat = vocab.compute_distance_matrix(D_raw, offset_value=OFFSET)

df = fix_clinical_diagnosis(dff.copy())
df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
df.sort_values('task_number_int', ascending=True, inplace=True)
df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
df.drop_duplicates(subset=['trigram'], keep='first', inplace=True)
df = df.reset_index(drop=True)
X_cat = np.vstack([upsample_sequence(np.asarray(s), L)
                   for s in df['int_cat_segm_embedding_labels']]).astype(int)
labels = df['pathologie'].values.astype(object)
G28 = D_G_cat.shape[0]
print('cohort:', df.groupby('pathologie').size().to_dict(), '| X_cat', X_cat.shape)

# group barycenters (mode-DBA) on the full cohort, for the audits
GROUPS = ['HEALTHY', 'RIL', 'TBI']
bary_mode = {g: B.barycenter_mode_dba(X_cat[labels == g], D_G_cat, nu=NU, lmbda=LMBDA) for g in GROUPS}
def hist(x, G=G28):
    h = np.bincount(np.asarray(x).astype(int), minlength=G).astype(float); return h / h.sum()


## Audit A — `p_match` vs symbol-frequency overlap (is p_match just a frequency statistic?)

In [ ]:
# For every (sequence, group-barycenter) pair: rTWE p_match vs histogram overlap.
from scipy.stats import pearsonr, spearmanr
pm, ho = [], []
for g in GROUPS:
    bh = hist(bary_mode[g])
    for x in X_cat:
        pm.append(B.pmatch_to_barycenter(x, bary_mode[g], D_G_cat, nu=NU, lmbda=LMBDA))
        ho.append(1.0 - 0.5*np.abs(hist(x) - bh).sum())   # 1 - 0.5*L1 = histogram overlap in [0,1]
pm, ho = np.array(pm), np.array(ho)
r_p = pearsonr(pm, ho)[0]; r_s = spearmanr(pm, ho)[0]
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.scatter(ho, pm, s=8, alpha=0.3)
ax.set_xlabel('histogram overlap (1 - 0.5 L1)'); ax.set_ylabel('rTWE p_match to barycenter')
ax.set_title(f'p_match vs frequency overlap\nPearson r={r_p:.2f}, Spearman r={r_s:.2f}')
plt.tight_layout(); plt.savefig(os.path.join(AUD, 'audit_pmatch_vs_histoverlap.png'), dpi=130); plt.show()
print(f'p_match correlates with pure frequency overlap: Pearson r={r_p:.3f}, Spearman r={r_s:.3f}')


## Audit B — fraction of edit (non-diagonal) moves in the rTWE alignment vs λ

In [ ]:
# Few edits => near-lockstep alignment => p_match ~ positional/Hamming overlap.
lam_grid = [0.0, 1e-3, 1e-2, 0.05, 0.1, 0.2, 0.5, 1.0]
rng = np.random.RandomState(0)
pairs = [(X_cat[i], bary_mode[g]) for g in GROUPS for i in rng.choice(len(X_cat), 15, replace=False)]
edit_frac = []
for lam in lam_grid:
    fr = []
    for s, b in pairs:
        path, _ = rtwe_alignment_path(s.astype(float), b.astype(float), D_G_cat, nu=NU, lmbda=lam)
        steps = [(path[k][0]-path[k-1][0], path[k][1]-path[k-1][1]) for k in range(1, len(path))]
        diag = sum(1 for di, dj in steps if di == 1 and dj == 1)
        fr.append(1.0 - diag/max(len(steps), 1))
    edit_frac.append(np.mean(fr))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lam_grid, edit_frac, 'o-')
ax.axvline(LMBDA, ls='--', c='red', label=f'selected λ={LMBDA}')
ax.set_xscale('symlog', linthresh=1e-3); ax.set_xlabel('λ (edit penalty)'); ax.set_ylabel('edit-move fraction')
ax.set_title('rTWE alignment: fraction of non-diagonal (edit) moves vs λ'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(AUD, 'audit_editfrac_vs_lambda.png'), dpi=130); plt.show()
print('edit-move fraction at selected λ=%.2f: %.3f' % (LMBDA, edit_frac[lam_grid.index(LMBDA)]))


## Audit C — AUC of histogram vs barycenter+p_match across λ (does any λ let alignment cross the histogram?)

In [ ]:
def cheap_auc_by_comp(methods, n_splits=3, rs=7, comp='HEALTHY_vs_RIL', pooled=False):
    lab = make_patient_control_labels(labels) if pooled else labels
    r = B.evaluate_baselines(X_cat, lab, methods, n_splits=n_splits, n_inits=1, random_state=rs)
    sub = r[r['comparison'] == comp]
    return sub['auc'].mean() if len(sub) else np.nan

hist_methods = {'wasserstein': B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['wasserstein']}
auc_hist = cheap_auc_by_comp(hist_methods)
auc_bary = []
for lam in lam_grid:
    m = {'tw_twe_mode': {'build': lambda X, seed, lam=lam: B.barycenter_mode_dba(X, D_G_cat, nu=NU, lmbda=lam),
                          'distance': lambda s, b, lam=lam: B.dist_neg_pmatch(s, b, D_G_cat, nu=NU, lmbda=lam)}}
    auc_bary.append(cheap_auc_by_comp(m))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lam_grid, auc_bary, 'o-', label='TW-TWE mode + p_match')
ax.axhline(auc_hist, ls='--', c='green', label=f'histogram (λ-independent) = {auc_hist:.2f}')
ax.set_xscale('symlog', linthresh=1e-3); ax.set_xlabel('λ'); ax.set_ylabel('HEALTHY_vs_RIL AUC (3 splits)')
ax.set_title('Histogram vs barycenter+p_match across λ'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(AUD, 'audit_auc_vs_lambda.png'), dpi=130); plt.show()
print('histogram AUC=%.2f ; best barycenter AUC over λ=%.2f' % (auc_hist, np.nanmax(auc_bary)))


## Audit D — barycenter symbol entropy (mode-collapse) vs pooled group entropy

In [ ]:
def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]; return float(-(p*np.log2(p)).sum())
rows = []
for g in GROUPS:
    pooled = hist(np.hstack(X_cat[labels == g]))
    rows.append({'group': g, 'barycenter': entropy(hist(bary_mode[g])), 'pooled sequences': entropy(pooled)})
edf = pd.DataFrame(rows).set_index('group')
display(edf)
ax = edf.plot(kind='bar', figsize=(6.5, 4.2)); ax.set_ylabel('Shannon entropy (bits)')
ax.set_title('Mode-DBA barycenter vs pooled-sequence symbol entropy')
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(os.path.join(AUD, 'audit_barycenter_entropy.png'), dpi=130); plt.show()


## Audit E — reconcile the handoff's "mean-DBA is broken" claim

Compare four barycenter constructions: **stock-aeon mean** (no D_G in averaging — the handoff's variant),
**mean-rTWE** (D_G passed into averaging, the thesis fork's intent), **mode**, and **D_G-Fréchet**.
Does passing D_G into the averaging change the verdict?

In [ ]:
from aeon.clustering.averaging import elastic_barycenter_average
def aeon_mean(Xg):
    try:
        b = elastic_barycenter_average(Xg[:, None, :].astype(float), distance='twe', nu=NU, lmbda=LMBDA)
        b = np.asarray(b).ravel()
    except Exception as e:
        print('aeon mean failed:', e); b = Xg[0].astype(float)
    return np.clip(np.rint(b), 0, G28-1).astype(int)

variants = {
    'aeon_mean(no D_G)': lambda X, seed: aeon_mean(X),
    'mean_rtwe(round)':  lambda X, seed: B.barycenter_mean_rtwe_dba(X, D_G_cat, nu=NU, lmbda=LMBDA, init='random', project='round', random_state=seed),
    'mode':              lambda X, seed: B.barycenter_mode_dba(X, D_G_cat, nu=NU, lmbda=LMBDA),
    'dg_frechet':        lambda X, seed: B.barycenter_mean_rtwe_dba(X, D_G_cat, nu=NU, lmbda=LMBDA, init='random', project='dg', random_state=seed),
}
rows = []
for name, build in variants.items():
    m = {name: {'build': build, 'distance': lambda s, b: B.dist_neg_pmatch(s, b, D_G_cat, nu=NU, lmbda=LMBDA)}}
    auc = cheap_auc_by_comp(m, n_splits=5)
    ent = np.mean([entropy(hist(build(X_cat[labels == g], 0))) for g in GROUPS])
    rows.append({'barycenter': name, 'HEALTHY_vs_RIL AUC': round(auc, 3), 'mean barycenter entropy': round(ent, 2)})
display(pd.DataFrame(rows))


## Audit synthesis

Reading A–E together: p_match is largely a frequency-overlap statistic (A); at the selected λ the rTWE
alignment is near-lockstep (B); no λ lets the barycenter cross the histogram (C); mode-DBA barycenters
collapse toward low-entropy frequent categories (D); and passing D_G into the averaging (mean-rTWE) does not
rescue the index-mean — mode/D_G-Fréchet are the categorically-meaningful constructions (E). The group signal
lives in *which categories occur*, which the histogram captures directly.

# Deliverable 3 — Experiments to beat majority voting

Success criterion: a method beats `majority_voting` AUC with a **BH-significant paired Wilcoxon**
(`baseline_significance_tests`, reference = majority_voting) on a named comparison. Reported honestly.

In [ ]:
# --- Exp (b): re-select (nu, lambda, offset) FOR THE p_match feature (anti-circular: select on
#     seeds 0-? , evaluate on disjoint seeds). Objective = mean pairwise AUC of tw_twe_mode + p_match. ---
grid = [(nu, lm, off) for nu in [0.0, 1e-4, 1e-3] for lm in [0.05, 0.1, 0.2, 0.5] for off in [0.1, 0.3]]
def sel_objective(nu, lm, off, rs):
    DGo = vocab.compute_distance_matrix(D_raw, offset_value=off)
    m = {'m': {'build': lambda X, seed: B.barycenter_mode_dba(X, DGo, nu=nu, lmbda=lm),
               'distance': lambda s, b: B.dist_neg_pmatch(s, b, DGo, nu=nu, lmbda=lm)}}
    r = B.evaluate_baselines(X_cat, labels, m, n_splits=3, n_inits=1, random_state=rs)
    return r.groupby('comparison')['auc'].mean().mean()
scores = [(sel_objective(nu, lm, off, rs=0), nu, lm, off) for (nu, lm, off) in grid]
sel = max(scores, key=lambda t: t[0])
sel_auc, SEL_NU, SEL_LM, SEL_OFF = sel
print(f'selected (selection-AUC={sel_auc:.3f}): nu={SEL_NU}, lambda={SEL_LM}, offset={SEL_OFF}')
D_G_sel = vocab.compute_distance_matrix(D_raw, offset_value=SEL_OFF)
sel_df = pd.DataFrame([{'sel_auc': s, 'nu': n, 'lambda': l, 'offset': o} for s, n, l, o in scores])
sel_df.sort_values('sel_auc', ascending=False).to_csv(os.path.join(EXP, 'hp_pmatch_selection_g28.csv'), index=False)
display(sel_df.sort_values('sel_auc', ascending=False).head(6))


In [ ]:
# --- Main beat-MV comparison (evaluation seeds DISJOINT from selection seed) ---
EVAL_RS = 1234   # disjoint from selection rs=0
methods = {
    'wasserstein': B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['wasserstein'],
    'majority_voting': B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['majority_voting'],
    'tw_twe_mode': {'build': lambda X, seed: B.barycenter_mode_dba(X, D_G_cat, nu=NU, lmbda=LMBDA),
                    'distance': lambda s, b: B.dist_neg_pmatch(s, b, D_G_cat, nu=NU, lmbda=LMBDA)},
    'tw_twe_reselected': {'build': lambda X, seed: B.barycenter_mode_dba(X, D_G_sel, nu=SEL_NU, lmbda=SEL_LM),
                          'distance': lambda s, b: B.dist_neg_pmatch(s, b, D_G_sel, nu=SEL_NU, lmbda=SEL_LM)},
}
# experimental methods (transition + shape_dba fast; eshape kept for a reduced-budget pass)
extra = B.extra_experiment_methods(D_G_cat, G28, nu=NU, lmbda=LMBDA, step_sequ=4)
methods['transition'] = extra['transition']
methods['shape_dba'] = extra['shape_dba']

res = []
res.append(B.evaluate_baselines(X_cat, labels, methods, n_splits=10, n_inits=3, random_state=EVAL_RS))
res.append(B.evaluate_baselines(X_cat, make_patient_control_labels(labels), methods, n_splits=10, n_inits=3, random_state=EVAL_RS))
# eshape_dtw at a reduced budget (O(L^2) pure-Python); pairwise only
res.append(B.evaluate_baselines(X_cat, labels, {'eshape_dtw': extra['eshape_dtw']}, n_splits=3, n_inits=1, random_state=EVAL_RS))
exp_all = pd.concat(res, ignore_index=True)
exp_all.to_csv(os.path.join(EXP, 'beat_mv_comparison_g28.csv'), index=False)
print('done; methods:', sorted(exp_all['method'].unique()))


In [ ]:
# --- table + significance vs majority voting ---
THREE = ['HEALTHY_vs_RIL', 'RIL_vs_TBI', 'CONTROL_vs_PATIENT']
agg = exp_all.groupby(['method', 'comparison'])['auc'].agg(['mean', 'std']).reset_index()
agg['cell'] = agg.apply(lambda r: f"{r['mean']:.2f}±{r['std']:.2f}", axis=1)
tab = agg.pivot(index='method', columns='comparison', values='cell')
order = ['majority_voting', 'wasserstein', 'transition', 'eshape_dtw', 'shape_dba',
         'tw_twe_mode', 'tw_twe_reselected']
display(tab.reindex([m for m in order if m in tab.index])[[c for c in THREE if c in tab.columns]])
sig = B.baseline_significance_tests(exp_all, reference='majority_voting')
sig.to_csv(os.path.join(EXP, 'beat_mv_significance_g28.csv'), index=False)
wins = sig[(sig['comparison'].isin(THREE)) & (sig['delta'] < 0) & (sig['significant'])]
print('Methods that BH-significantly BEAT majority voting (delta<0 vs MV reference):')
display(wins if len(wins) else 'NONE — no experimental method significantly beat majority voting.')


In [ ]:
# --- FIGURE: experiments vs majority voting ---
piv = exp_all.groupby(['method', 'comparison'])['auc'].mean().unstack()
piv = piv.reindex([m for m in order if m in piv.index])
fig, ax = plt.subplots(figsize=(12, 5))
piv[[c for c in THREE if c in piv.columns]].plot(kind='bar', ax=ax, width=0.8)
mv = piv.loc['majority_voting'] if 'majority_voting' in piv.index else None
ax.axhline(0.5, ls='--', c='gray', lw=1)
ax.set_ylabel('AUC'); ax.set_ylim(0.3, 1.0); ax.set_title('Beat-majority-voting experiments (G=28)')
ax.legend(title='comparison', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(EXP, 'beat_mv_auc_bars.png'), dpi=130); plt.show()


In [ ]:
# --- Exp (d): barycenter-length sweep + mean-vs-mode ablation ---
len_rows = []
for Lx in [32, 64, 128]:
    Xx = np.vstack([upsample_sequence(np.asarray(s), Lx) for s in df['int_cat_segm_embedding_labels']]).astype(int)
    msets = {
        'wasserstein': B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['wasserstein'],
        'majority_voting': B.default_baseline_methods(D_G_cat, nu=NU, lmbda=LMBDA)['majority_voting'],
        'tw_twe_mode': {'build': lambda X, seed: B.barycenter_mode_dba(X, D_G_cat, nu=NU, lmbda=LMBDA),
                        'distance': lambda s, b: B.dist_neg_pmatch(s, b, D_G_cat, nu=NU, lmbda=LMBDA)},
        'tw_twe_mean': {'build': lambda X, seed: B.barycenter_mean_rtwe_dba(X, D_G_cat, nu=NU, lmbda=LMBDA, init='random', project='round', random_state=seed),
                        'distance': lambda s, b: B.dist_neg_pmatch(s, b, D_G_cat, nu=NU, lmbda=LMBDA)},
    }
    r = B.evaluate_baselines(Xx, labels, msets, n_splits=3, n_inits=1, random_state=7)
    for m in msets:
        len_rows.append({'L': Lx, 'method': m,
                         'HEALTHY_vs_RIL': r[(r.method == m) & (r.comparison == 'HEALTHY_vs_RIL')]['auc'].mean()})
ldf = pd.DataFrame(len_rows).pivot(index='L', columns='method', values='HEALTHY_vs_RIL')
display(ldf)
ax = ldf.plot(marker='o', figsize=(7, 4.5)); ax.set_ylabel('HEALTHY_vs_RIL AUC'); ax.set_xlabel('barycenter length L')
ax.set_title('Length sweep + mean-vs-mode ablation (G=28)')
plt.tight_layout(); plt.savefig(os.path.join(EXP, 'length_meanmode_ablation.png'), dpi=130); plt.show()


## Honest conclusion

**No experimental method BH-significantly beat `majority_voting`** (significance table above). Observed AUC
(10x3, evaluation seeds disjoint from HP selection):

| method | RIL-vs-Ctrl | TBI-vs-RIL | Patient-vs-Ctrl |
|---|---|---|---|
| majority voting | 0.71 | 0.53 | 0.59 |
| histogram | 0.77 | 0.56 | 0.76 |
| transition (bigram ordering) | 0.62 | 0.44 | 0.62 |
| Edit-Shape DTW | 0.53 | **0.59** | – |
| soft-mode DBA | 0.63 | 0.52 | 0.59 |
| TW-TWE mode (re-selected HP ν=1e-3,λ=0.2) | 0.65 | 0.52 | 0.60 |

**Audit (why):** p_match is substantially a frequency-overlap statistic (Pearson r=0.63, Audit A); the
alignment *does* warp (~44% edit moves at λ=0.1, Audit B) but no λ lets the barycenter cross the histogram
(Audit C: 0.74 vs 0.66); mode-DBA barycenters are mode-collapsed (lower entropy than the pooled sequences,
Audit D); and passing D_G into the *averaging* does **not** rescue the index-mean — `mean_rtwe` (0.45) ≈
`aeon_mean` (0.54), both near chance, while `mode` (0.65) / `dg_frechet` (0.66) are the categorically
meaningful constructions (Audit E). The barycenter discards the precise per-category frequencies the
histogram keeps, and the temporal structure it adds is not group-discriminative.

**One hint:** Edit-Shape DTW is the best method on **TBI-vs-RIL (0.59)** — the single comparison where
frequency is weakest (histogram 0.56, MV 0.53) — suggesting temporal-ordering carries marginal signal there,
but **not BH-significant**.

**Verdict (reported honestly, no p-hacking):** on this G=28 representation the frequency baselines win;
alignment/ordering does not beat them. The contribution should be reframed per
`RESULTS_HANDOFF_barycenters.md` §8, not tuned to invert.